# HINT-debug: variable-driven postprocessing

Tracing retains its adaptive tolerances; repeated sampling reuses compiled kernels with fresh field values. Completed checkpoints with missing physical data are rejected. Use a new analysis output when replacing the source grid or wall.

Unexecuted template: replace the paths and probe positions. Three categories are **1-D profiles / time histories**, **2-D field sections**, and **Poincare sections**. Install `python -m pip install -e './HINT-debug[plots]'`. The analysis CLI still writes a separate NetCDF; these notebook helpers return figures and numerical arrays without modifying the main result.


Schema 9 stores A0/A1 and derives magnetic fields with the evolution curl. Plotting calls and SI units are unchanged. Only current A-state files are accepted for analysis and follow.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from hint_debug_plotting import HintPlots, variable_catalog

case = Path("/path/to/case")
plots = HintPlots(
    case / "hint_debug.nc",
    record=-1,  # Last complete checkpoint; alternatively use outer_step=182, not both.
    wout=case / "wout_case.nc",  # Optional VMEC geometry/field/profile reference.
    backend="cpu",  # "cpu" or "gpu"; does not change the scientific definition.
    engine="jax",  # "jax" or "native"; plotting itself remains on CPU.
)
figures = case / "figures"
# Use a stopped result or stable snapshot, never an actively written file.
# No coil text or separate wall file is needed; wall/coil field are stored in main.nc.
# These probe the continuous curl(A) used for tracing, not the grid derivatives
# plotted under divergence_b*. Setup may reconstruct transient A0/A1 splines.
print(plots.magnetic_interpolation_diagnostics(vacuum=True, maximum=2048))
print(plots.magnetic_interpolation_diagnostics(vacuum=False, maximum=2048))


## Variable names and definitions

Names are case-sensitive strings. Spatial variables support profiles, sections and checkpoint time histories. Aggregate metrics are not 2-D fields. The catalog below is also listed in `post.toml` comments. Units are SI except explicitly stored normalized diagnostics. Derivative-invalid wall nodes and undefined ratios remain gaps, not zeros.


In [ ]:
# Notebook plotting variable catalog (case-sensitive strings, not TOML keys).
# Spatial variables below work with profiles(), sections(), and time_series().
# toroidal_current is a profile-only s/rho integral; stored metrics and maxima
# are time-series-only. See postprocess_demo.ipynb for actual calls.
# profiles: R at fixed Z/phi, Z at fixed R/phi, phi at fixed R/Z, or s/rho bins.
# time_series: fields use R-volume-weighted RMS by default; reduction='mean',
# 'min', 'max', or 'rms' overrides this. point=(R_m,Z_m,phi_rad) samples a point.
# Stored Step-B metrics retain solver normalization. Spatial derivatives are SI.
# scope='wall' (default) or 'plasma' (inside wall with saved evolving 0<=s<1).
# pressure_max is a sampled grid maximum, not a magnetic-axis reconstruction.
# Rates use differences between saved complete checkpoints / converted relaxation
# seconds, not physical acceleration or wall-clock time. Missing history is NaN.
# Local force ratio uses max(|J_response x B_total|,|grad(p)|), range [0,2].
# Denominators <=1e-10 of their valid-domain maximum are masked, not replaced.
# pressure: Pa; Pressure.
# s: 1; Evolving s label.
# rho: 1; sqrt(s).
# field_strength: T; Total field strength.
# response_field_strength: T; Response field strength.
# vacuum_field_strength: T; Vacuum field strength.
# speed: m/s; Speed.
# velocity_change_rate: m/s^2; |v-v_previous|/delta_t; converted relaxation seconds, not
#   physical acceleration.
# speed_change_rate: m/s^2; (|v|-|v_previous|)/delta_t; signed secant in converted relaxation
#   seconds.
# force_residual: N/m^3; Force residual magnitude.
# force_residual_relative: 1; |J1 x B-grad(p)|/max(|J1 x B|,|grad(p)|); undefined if both forces
#   vanish.
# lorentz_force: N/m^3; Lorentz force magnitude.
# pressure_gradient: Pa/m; Pressure gradient magnitude.
# parallel_pressure_gradient: Pa/m; signed b.grad(p), b=B/|B|.
# current_density: A/m^2; Response current density magnitude.
# toroidal_current_density: A/m^2; signed pointwise J_phi, not VMEC jcurv or dI/ds.
# parallel_current_density: A/m^2; Parallel response current density.
# field_r: T; Total B r.
# field_phi: T; Total B phi.
# field_z: T; Total B z.
# response_field_r: T; Response B r.
# response_field_phi: T; Response B phi.
# response_field_z: T; Response B z.
# velocity_r: m/s; Velocity r.
# velocity_phi: m/s; Velocity phi.
# velocity_z: m/s; Velocity z.
# divergence_b: T/m; div(total B).
# divergence_b_abs: T/m; |div(total B)|.
# divergence_b_response: T/m; div(response B).
# divergence_b_response_abs: T/m; |div(response B)|.
# divergence_b_vacuum: T/m; div(vacuum B).
# divergence_b_vacuum_abs: T/m; |div(vacuum B)|.
# toroidal_current: A; signed integral of J_phi dR dZ below an s/rho threshold; profiles only.
# rotational_transform: 1; trace-derived profile, not a local 2-D field or saved history.
# Use plots.rotational_transform(crossings=128,toroidal_steps=128) with wout,
# or explicit R-Z seeds. Mask failed traces; inspect finite-window/surface flags.
# wout= on HintPlots enables axis + s=.25,.5,.75 + LCFS geometry, VMEC pressure,
# total-B and current references. s profiles compare each dataset's own labels;
# R/Z/phi profiles sample identical spatial positions. No exterior extrapolation.
# kinetic_energy: normalized; Kinetic energy; stored Step-B diagnostic, not a recomputed
#   checkpoint field.
# response_magnetic_energy: normalized; Response magnetic energy; stored Step-B diagnostic, not
#   a recomputed checkpoint field.
# force_max: normalized; Force max; stored Step-B diagnostic, not a recomputed checkpoint field.
# force_rms: normalized; Force RMS; stored Step-B diagnostic, not a recomputed checkpoint field.
# divb_max: normalized; div(B_response) max; stored Step-B diagnostic, not a recomputed
#   checkpoint field.
# divb_rms: normalized; div(B_response) RMS; stored Step-B diagnostic, not a recomputed
#   checkpoint field.
# force_volume_rms: normalized; Force volume-weighted RMS; stored Step-B diagnostic, not a
#   recomputed checkpoint field.
# divb_volume_rms: normalized; div(B_response) volume-weighted RMS; stored Step-B diagnostic,
#   not a recomputed checkpoint field.
# parallel_pressure_volume_rms: normalized; b.grad(p) volume-weighted RMS; stored Step-B
#   diagnostic, not a recomputed checkpoint field.
# boundary_bn_max: normalized; Frozen normal-field error max; stored Step-B diagnostic, not a
#   recomputed checkpoint field.
# boundary_velocity_max: normalized; Box boundary speed max; stored Step-B diagnostic, not a
#   recomputed checkpoint field.
# pressure_max: Pa; maximum over valid saved grid nodes in the selected scope; not an axis
#   solve.
# field_max: T; maximum over valid saved grid nodes in the selected scope; not an axis solve.
# speed_max: m/s; maximum over valid saved grid nodes in the selected scope; not an axis solve.
# Plot helpers return figures/data and never change this TOML or the main NetCDF.
# These names do not enable plots in the numerical CLI: use the notebook API.

catalog = variable_catalog()
print(list(plots.variables(kind="section")))
print(catalog["force_residual_relative"])


## 1-D: profiles

All angles are radians, positions are metres. Defaults are the two endpoints and midpoint of half a field period: `0, pi/(2*nfp), pi/nfp`. Supply `angles=[...]` for other cuts. Line interpolation is linear in R/Z and periodic in phi; components are interpolated before vector norms or local ratios. Higher plotting DPI does not improve the underlying mesh resolution.

VMEC references use their own flux labels. R/Z/phi lines evaluate the same physical positions by inverting the supplied wout geometry, with NaN outside its LCFS. Pressure is raw presf; total B comes from Fourier coefficients; local J_phi is curl(B)/mu0 and enclosed I is the signed Ampere integral, not jcurv. Set compare_vmec=False to hide numerical comparisons.


In [ ]:
# Replace these illustrative probe positions with locations inside your wall.
r_probe = float(plots.grid.r.mean())
z_probe = float(plots.grid.z.mean())

radial = plots.profiles("pressure", coordinate="R", z_m=z_probe)
radial.save(figures / "pressure_R.png", dpi=300)
plt.close(radial.figure)

vertical = plots.profiles("field_strength", coordinate="Z", r_m=r_probe)
vertical.save(figures / "field_Z.png", dpi=300)
plt.close(vertical.figure)

# phi is the horizontal coordinate here, so do not also pass angles.
toroidal = plots.profiles("speed", coordinate="phi", r_m=r_probe, z_m=z_probe)
toroidal.save(figures / "speed_phi.png", dpi=300)
plt.close(toroidal.figure)

# Evolving saved s, NOT the initial VMEC surface map. Per-section area-bin
# means +/- one within-bin std, NOT true flux-surface averages.
profile = plots.profiles("pressure", coordinate="s", bins=50, min_count=2)
profile.save(figures / "pressure_s.png", dpi=300)
plt.close(profile.figure)
# coordinate="rho" uses sqrt(s). Pressure overlays unscaled wout presf if provided,
# otherwise the stored scaled target. No reference is fitted to the evolved state.
current = plots.profiles("toroidal_current", coordinate="rho", bins=50)
current.save(figures / "enclosed_current_rho.png", dpi=300)
plt.close(current.figure)
# toroidal_current integrates J_phi*dR*dZ below each saved-s threshold;
# toroidal_current_density is local signed J_phi, not VMEC jcurv or dI/ds.


## 1-D: time histories

`time_series` reads only completed records through the selected checkpoint. Stored Step-B metrics preserve their original scope and normalization; recomputed checkpoint fields use physical units. A checkpoint is after any enabled outer pressure feedback, so its force need not equal the final Step-B sample. With solver.scale_after=false, there is no post-ramp amplitude feedback (from the first outer cycle in a VMEC start). Stored pressure_scale=1 denotes a completed ramp schedule, not constant actual axis pressure. `x='time'` is normalized relaxation time; `x='time_s'` is converted relaxation time, **not elapsed runtime or physical experimental time**. `pressure_max` is a grid maximum, not an axis solve. The ramp marker uses saved ramp settings unless explicitly overridden.


In [ ]:
history = plots.time_series(
    ["force_rms", "divb_rms", "boundary_bn_max", "pressure_max", "speed_max"],
    x="outer_step", scale="auto",
    # ramp_end=20,  # Optional explicit marker; default uses stored ramp length.
)
history.save(figures / "convergence.png", dpi=300)
plt.close(history.figure)

# Field reductions: volume weights R*dR*dZ*dphi, valid nodes only.
relative = plots.time_series(
    ["force_residual_relative", "velocity_change_rate", "divergence_b_response_abs"],
    reduction="rms", scope="plasma", x="time",
)
relative.save(figures / "field_rms_history.png", dpi=300)
plt.close(relative.figure)

# Fixed Eulerian point. speed_change_rate is signed; it is not |delta v|/delta t.
point_history = plots.time_series(
    ["pressure", "speed", "speed_change_rate"],
    point=(r_probe, z_probe, 0.0), x="time_s", scale="auto",
)
point_history.save(figures / "point_history.png", dpi=300)
plt.close(point_history.figure)
# Missing predecessor checkpoints produce NaN rates. A point outside the chosen
# mask at any saved step also gives NaN. No time-interpolated states are invented.


## 2-D: toroidal field sections

Each figure vertically stacks three `contourf + contour` sections with a common color scale, the initial VMEC LCFS, stored first-wall contour and rectangular computational box. `scope='plasma'` selects evolved `0<=s<1` inside the wall; default `scope='wall'` includes vacuum inside the wall even for HINT-debug. Force diagnostics use `J_response x B_total - grad(p)`, not Jnet added again. The local ratio divides by `max(|J_response x B_total|,|grad(p)|)`; a vanishing/tiny denominator is masked. A vacuum point with negligible pressure gradient may have ratio 1 despite small absolute force, so inspect both absolute and relative plots.

The VMEC magnetic axis, s=.25,.5,.75 surfaces and LCFS are overlaid by default. Magenta dashed same-value contours compare pressure, total B and J_phi when available. No velocity/force/divergence reference is fabricated from VMEC.


In [ ]:
for quantity in ("pressure", "field_strength", "speed", "force_residual",
                 "force_residual_relative", "divergence_b_response_abs"):
    result = plots.sections(quantity, levels=60, contour_lines=12,
                            scale="auto", scope="wall")
    result.save(figures / f"{quantity}_sections.png", dpi=300)
    plt.close(result.figure)
# Signed fields/divergence: use linear or symlog to retain signs and zeros.
# force_residual_relative is dimensionless; divergence_b* is T/m, NOT the
# normalized divb_rms/divb_max copied from Step-B diagnostic history.


## Special: Poincare sections

Seeds cover the first-wall interior, not only the LCFS. Increase density or provide R/Z ranges for island searches. Chunking bounds temporary memory; it does not reduce requested return crossings. Tracing uses the selected CPU/GPU/JAX backend. The notebook API is single process; the existing numerical analysis CLI retains MPI support. Each section is seeded independently. Gaps may be unresolved or wall-terminated traces, not proof of absent islands.


In [ ]:
crossings = plots.poincare_data(
    seed_shape=(32, 40), crossings=400, toroidal_steps=128,
    field_source="hint", both_directions=False, chunk_crossings=8,
    rtol=1e-8, atol=1e-10,
    # r_range=(...), z_range=(...),  # Optional metre bounds inside stored grid.
    # seeds=[[R0, Z0], [R1, Z1]],  # Optional explicit seeds instead of a grid.
)
result = plots.poincare(data=crossings, marker_size=0.10,
                        vmec_surfaces=[0, .25, .5, .75, 1])
result.save(figures / "poincare.png", dpi=300)
plt.close(result.figure)
# Reuse crossings to restyle without retracing. status: 0 valid, 1 wall,
# 2 singular field, 3 unresolved. No automatic output NetCDF is written here.


## Rotational transform with VMEC comparison

Use dense substep winding about the **evolved HINT magnetic axis**, not sparse Poincare returns. VMEC orientation and integer*nfp poloidal-frame winding are accounted for. VMEC iotaf (or non-dummy iotas half mesh) is the reference. HINT s at the launch point remains an evolving label, not a new measured toroidal flux. Finite-length difference is not a rigorous error bound. Filled points pass the trace-half and surface-consistency tests; hollow points are unresolved. Failed or underresolved traces remain NaN. A rational/island/chaotic trace must not be silently interpreted as a nested surface. Inspect Poincare together with this plot.


In [ ]:
iota_data = plots.rotational_transform_data(
    seed_s=[.05, .15, .3, .5, .7, .9],  # VMEC launch surfaces; seed_theta=0
    crossings=128, toroidal_steps=128, chunk_crossings=4,
    convergence_tolerance=.002, surface_tolerance=.02,
    rtol=1e-8, atol=1e-10,
    # seeds=[[R0,Z0], ...],  # Use instead of seed_s; metres, required without wout.
    # axis_seed=[R_axis,Z_axis],  # Optional evolved-axis seed at phi=0, metres.
)
for coordinate in ("s", "rho", "R"):
    result = plots.rotational_transform(data=iota_data, coordinate=coordinate)
    result.save(figures / f"iota_{coordinate}.png", dpi=300)
    plt.close(result.figure)
# String API: plots.profiles("rotational_transform", trace_options={"crossings":128})
print(iota_data[0]["half_window_difference"])
print(iota_data[0]["converged"], iota_data[0]["surface_resolved"])
